# Award recipients: organisations, countries and regions

One side table for the funding-flows release (#1570). Geography retains the existing place-of-performance priority.
Organisations: USAspending x4, IDRC, IATI (all role-2/role-4 partners, including child activities), Gates and Wellcome.
IATI funding/extending organisations are excluded. Organisation country comes from its identifier, never the project country.
Missing organisation shares remain null; organisation roles do not form a geographic allocation.
Rerun this producer when any source refreshes; publish the side table before the single nightly release.
USAspending recipients explicitly typed Individual (business type P) are excluded from organisations; geographic evidence remains.


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.award_recipients_raw AS
WITH geography AS (
SELECT abs(xxhash64(concat(b.funder_id, ':', lower(b.funder_award_id)))) % 9000000000 AS staging_id,
       b.funder_id, lower(b.funder_award_id) AS award_key, b.provenance, b.funder_award_id,
       b.kind, b.country_code, b.region_code, b.region_vocabulary, b.basis, sum(b.share) AS share,
       CASE WHEN max(b.share) IS NOT NULL THEN 'funder_reported' END AS share_basis, b.source_field,
       'funder' AS asserted_by, 'af1570-phase1-v1' AS adapter_version, current_timestamp() AS asserted_at
FROM (
WITH iso AS (SELECT upper(alpha_3) a3, upper(id) a2 FROM openalex.common.countries WHERE alpha_3 IS NOT NULL),
us AS (
  SELECT 'usaspending_usaid' provenance, 4320306115L funder_id, award_id_fain fain, recipient_country_code rc, primary_place_of_performance_country_code pc FROM openalex.awards.usaid_awards_raw
  UNION ALL SELECT 'usaspending_cdc', 4320332162L, award_id_fain, recipient_country_code, primary_place_of_performance_country_code FROM openalex.awards.cdc_awards_raw
  UNION ALL SELECT 'usaspending', 4320306078L, award_id_fain, recipient_country_code, primary_place_of_performance_country_code FROM openalex.awards.dod_awards_raw
  UNION ALL SELECT 'usaspending_usda', 4320306114L, award_id_fain, recipient_country_code, primary_place_of_performance_country_code FROM openalex.awards.usda_awards_raw
),
us_rows AS (
  SELECT provenance, funder_id, fain AS funder_award_id, 'country' kind, i.a2 country_code, CAST(NULL AS STRING) region_code, CAST(NULL AS STRING) region_vocabulary,
         'organisation_location' basis, CAST(NULL AS DOUBLE) share, 'recipient_country_code' source_field
  FROM us JOIN iso i ON upper(us.rc) = i.a3 WHERE fain IS NOT NULL
  UNION ALL
  SELECT provenance, funder_id, fain, 'country', i.a2, NULL, NULL, 'place_of_performance', CAST(NULL AS DOUBLE), 'primary_place_of_performance_country_code'
  FROM us JOIN iso i ON upper(us.pc) = i.a3 WHERE fain IS NOT NULL
),
idrc AS (
  SELECT iati_identifier,
         from_json(recipient_countries_json, 'array<struct<code:string,percentage:string>>') cs,
         from_json(recipient_regions_json, 'array<struct<code:string,percentage:string,vocabulary:string>>') rs
  FROM openalex.awards.idrc_raw
),
idrc_rows AS (
  SELECT 'idrc_iati' provenance, 4320319949L funder_id, iati_identifier funder_award_id, 'country' kind, upper(trim(c.code)) country_code, CAST(NULL AS STRING) region_code, CAST(NULL AS STRING) region_vocabulary,
         'funder_reported' basis, try_cast(c.percentage AS DOUBLE) / 100.0 share, 'recipient-country' source_field
  FROM idrc LATERAL VIEW explode(cs) e AS c
  UNION ALL
  SELECT 'idrc_iati', 4320319949L, iati_identifier, 'region', NULL, r.code, coalesce(r.vocabulary, '1'), 'funder_reported', try_cast(r.percentage AS DOUBLE) / 100.0, 'recipient-region'
  FROM idrc LATERAL VIEW explode(rs) e AS r
)
SELECT DISTINCT * FROM us_rows UNION ALL SELECT DISTINCT * FROM idrc_rows
) b
GROUP BY b.funder_id, b.funder_award_id, b.provenance, b.kind, b.country_code, b.region_code, b.region_vocabulary, b.basis, b.source_field
), organisations AS (
WITH country_aliases AS (
  -- Source spellings equivalent to the versioned country/award_country_lookup.csv ISO aliases.
  SELECT * FROM VALUES ('bolivia, plurinational state of','BO'),('bosnia-herzegovina','BA'),('congo, democratic republic of','CD'),('congo, the democratic republic of the','CD'),('cote d\'ivoire','CI'),('czech republic','CZ'),('côte d\'ivoire','CI'),('dominican republic','DO'),('korea, republic of','KR'),('palestine, state of','PS'),('palestinian territories','PS'),('taiwan, province of china','TW'),('tanzania, united republic of','TZ'),('trinidad and tobago','TT'),('türkiye','TR'),('venezuela, bolivarian republic of','VE'),('viet nam','VN') AS aliases(value,code)
), country_names AS (
  SELECT lower(trim(value)) value, min(upper(id)) code FROM openalex.common.countries
  LATERAL VIEW explode(array(id,alpha_3,display_name,full_name)) c AS value
  WHERE nullif(trim(value),'') IS NOT NULL GROUP BY 1 HAVING count(DISTINCT id)=1
), us AS (
  SELECT 'usaspending_usaid' provenance,4320306115L funder_id,award_id_fain funder_award_id,recipient_name name,recipient_country_code country FROM openalex.awards.usaid_awards_raw
    WHERE NOT coalesce(upper(trim(business_types_code))='P' OR upper(trim(business_types_description)) IN ('INDIVIDUAL','INDIVIDUALS'),false)
  UNION ALL SELECT 'usaspending_cdc',4320332162L,award_id_fain,recipient_name,recipient_country_code FROM openalex.awards.cdc_awards_raw
    WHERE NOT coalesce(upper(trim(business_types_code))='P' OR upper(trim(business_types_description)) IN ('INDIVIDUAL','INDIVIDUALS'),false)
  UNION ALL SELECT 'usaspending',4320306078L,award_id_fain,recipient_name,recipient_country_code FROM openalex.awards.dod_awards_raw
    WHERE NOT coalesce(upper(trim(business_types_code))='P' OR upper(trim(business_types_description)) IN ('INDIVIDUAL','INDIVIDUALS'),false)
  UNION ALL SELECT 'usaspending_usda',4320306114L,award_id_fain,recipient_name,recipient_country_code FROM openalex.awards.usda_awards_raw
    WHERE NOT coalesce(upper(trim(business_types_code))='P' OR upper(trim(business_types_description)) IN ('INDIVIDUAL','INDIVIDUALS'),false)
), iati AS (
  SELECT provenance,try_cast(openalex_funder_id AS BIGINT) funder_id,trim(funder_award_id) funder_award_id,reporting_org_ref,reporting_org_name,
    concat(coalesce(from_json(participating_orgs_json,'array<struct<role:string,ref:string,name:string,name_en:string,name_fr:string,name_es:string,percentage:string>>'),array()),
      coalesce(from_json(children_participating_orgs_json,'array<struct<role:string,ref:string,name:string,name_en:string,name_fr:string,name_es:string,percentage:string>>'),array())) orgs
  FROM openalex.awards.iati_raw
  WHERE ship='true' AND is_research='true' AND research_rule_ids IS NOT NULL
    AND NOT coalesce(try_cast(award_amount AS DOUBLE)<=1 OR lower(trim(title)) IN ('not in use','changed')
      OR lower(trim(title)) RLIKE '^(travel\\b|programme related costs)',false)
  UNION ALL
  SELECT 'idrc_iati',4320319949L,iati_identifier,CAST(NULL AS STRING),CAST(NULL AS STRING),
    from_json(participating_orgs_json,'array<struct<role:string,ref:string,name:string,name_en:string,name_fr:string,name_es:string,percentage:string>>')
  FROM openalex.awards.idrc_raw
), source_orgs AS (
  SELECT *, 'lead' role,CAST(NULL AS DOUBLE) share,CAST(NULL AS STRING) ror,'recipient_name' source_field FROM us
  UNION ALL
  SELECT 'gates_foundation',4320306137L,grant_id,grantee_name,grantee_country,'lead',NULL,NULL,'grantee_name' FROM openalex.awards.gates_raw
  UNION ALL
  SELECT 'wellcome_trust',4320311904L,grant_ref,recipient_org_name,recipient_orgcountry,'lead',NULL,
    CASE WHEN recipient_org_id RLIKE '^https://ror.org/[0-9a-z]{9}$' THEN recipient_org_id END,'recipient_org_name'
    FROM openalex.awards.wellcome_raw WHERE grant_ref NOT LIKE '360G-%'
  UNION ALL
  SELECT provenance,funder_id,funder_award_id,coalesce(o.name,o.name_en,o.name_fr,o.name_es),
    CASE WHEN o.ref RLIKE '^[A-Z]{2}-' THEN substring(o.ref,1,2) END,
    CASE o.role WHEN '2' THEN 'partner' WHEN '4' THEN 'implementing' END,
    try_cast(o.percentage AS DOUBLE)/100.0,
    CASE WHEN o.ref RLIKE '^https://ror.org/[0-9a-z]{9}$' THEN o.ref END,'participating-org'
  FROM iati LATERAL VIEW explode(orgs) e AS o WHERE o.role IN ('2','4')
    -- Implementers can also co-fund: retain role 4 even when the same organisation has role 1.
    -- Only remove the publisher/extending agency's self-accountability (role 2).
    AND (o.role='4' OR NOT (
      coalesce(nullif(trim(o.ref),'')=reporting_org_ref,false)
      OR coalesce(lower(trim(coalesce(o.name,o.name_en,o.name_fr,o.name_es)))=lower(trim(reporting_org_name)),false)
      OR coalesce(exists(orgs, f -> f.role='3' AND
        ((nullif(trim(o.ref),'') IS NOT NULL AND o.ref=f.ref)
         OR lower(trim(coalesce(o.name,o.name_en,o.name_fr,o.name_es)))=lower(trim(coalesce(f.name,f.name_en,f.name_fr,f.name_es))))),false)))
)
SELECT DISTINCT abs(xxhash64(concat(s.funder_id,':',lower(s.funder_award_id)))) % 9000000000 staging_id,
  s.funder_id,lower(s.funder_award_id) award_key,s.provenance,s.funder_award_id,
  'organisation' kind,coalesce(c.code,ca.code) country_code,CAST(NULL AS STRING) region_code,CAST(NULL AS STRING) region_vocabulary,
  'funder_reported' basis,s.share,CASE WHEN s.share IS NOT NULL THEN 'funder_reported' END share_basis,
  s.source_field,'funder' asserted_by,'af1570-organisations-v1' adapter_version,current_timestamp() asserted_at,
  trim(s.name) name,s.role,s.ror
FROM source_orgs s LEFT JOIN country_names c ON c.value=lower(trim(s.country))
LEFT JOIN country_aliases ca ON ca.value=lower(trim(s.country))
WHERE nullif(trim(s.name),'') IS NOT NULL AND nullif(trim(s.funder_award_id),'') IS NOT NULL

)
SELECT *,CAST(NULL AS STRING) name,CAST(NULL AS STRING) role,CAST(NULL AS STRING) ror FROM geography
UNION ALL SELECT * FROM organisations;


In [ ]:
%sql
SELECT provenance, kind, basis, count(*) rows, count(DISTINCT staging_id) awards
FROM openalex.awards.award_recipients_raw GROUP BY ALL ORDER BY ALL

In [ ]:
%sql
-- Shape the nightly checks (RECIPIENT_RAW_SHAPE / RECIPIENT_RAW_UNIQUE): nothing may come back.
SELECT * FROM openalex.awards.award_recipients_raw
WHERE NOT coalesce(length(provenance)>0 AND length(funder_award_id)>0 AND staging_id IS NOT NULL AND funder_id IS NOT NULL AND length(award_key)>0
  AND kind IN ('country','region','organisation') AND basis IN ('funder_reported','place_of_performance','organisation_location')
  AND asserted_by IN ('funder','openalex','curated') AND (share IS NULL OR share BETWEEN 0 AND 1)
  AND CASE kind WHEN 'country' THEN country_code RLIKE '^[A-Z]{2}$' AND region_code IS NULL
                WHEN 'region' THEN length(region_code)>0 AND length(region_vocabulary)>0 AND country_code IS NULL
                WHEN 'organisation' THEN length(trim(name))>0 AND role IN ('lead','partner','implementing','beneficiary','host')
                  AND (country_code IS NULL OR country_code RLIKE '^[A-Z]{2}$') AND region_code IS NULL END, false)
UNION ALL
SELECT * EXCEPT(dup) FROM (SELECT *, count(*) OVER (PARTITION BY provenance,staging_id,funder_id,award_key,kind,country_code,region_code,region_vocabulary,basis,name,role,ror) dup
  FROM openalex.awards.award_recipients_raw) WHERE dup > 1